# 16 — P2: MILP Encoding of Multi-Head Self-Attention

**Plan 2 — Phase 4.3.** Sound MILP encoding for one MHSA block:
$$\text{Attn}(x) = W_O \cdot \text{concat}_h \Big[\,\text{softmax}\big(\tfrac{Q_h K_h^T}{\sqrt{D_h}}\big)\, V_h\,\Big]$$

The encoding chains five stages, each individually sound:

1. **Linear Q, K, V projections** — exact linear constraints.
2. **Score matrix** `S[h,i,j] = (1/√D) Σ_d Q[h,i,d]·K[h,j,d]` — bilinear,
   sum-of-McCormick.
3. **Stability shift** — subtract per-row IBP-derived score-upper-bound max;
   safe because softmax is shift-invariant.
4. **Softmax**:
   - `E[h,i,j] = exp(S_shift)` — convex PWL bracket on `[s_lo, s_up]`.
   - `Sum_E[h,i] = Σ_j E[h,i,j]` — linear.
   - `Inv_E[h,i] = 1/Sum_E[h,i]` — convex PWL bracket on `[Σ e_lo, Σ e_up]`.
   - `A[h,i,j] = E[h,i,j] · Inv_E[h,i]` — McCormick bilinear, clamp to `[0,1]`.
5. **Output** `O[h,i,d] = Σ_j A[h,i,j]·V[h,j,d]` — McCormick (A bounded in [0,1]
   gives tight envelopes).
6. **Output projection** `W_O` — linear.

## IBP precomputation (mandatory)
PWL brackets need numerical input domains; McCormick needs interval bounds on
both factors. We pre-compute interval bounds for every intermediate quantity
by IBP and pass those into the encoder.

## Tests
Soundness on a tiny standalone MHSA layer (small `N`, `E`, `H` for tractability):
random weights, random input box, then for each output coordinate solve
`min/max O_{i,d}` and check the resulting MILP-certified interval brackets
the true PyTorch output for K random points in the box.

For a single block with `N=49, E=64, H=2` (full ViT-Tiny), this encoding
produces ~10⁴ bilinear products and is solved in notebook 17 onward.


In [2]:
!pip install -q numpy torch gurobipy

In [3]:
from __future__ import annotations
import math, json, time, warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Callable, List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    import gurobipy as gp
    from gurobipy import GRB
    HAS_GUROBI = True
except Exception as e:
    HAS_GUROBI = False
    print(f'Gurobi not available ({e}); MILP tests skipped.')

warnings.filterwarnings('ignore')
rng = np.random.default_rng(1234)
torch.manual_seed(1234)
print(f'NumPy {np.__version__}  Torch {torch.__version__}  Gurobi={HAS_GUROBI}')

NumPy 2.0.2  Torch 2.10.0+cu128  Gurobi=True


In [4]:
# ── PWL bracket primitives + Gurobi big-M encoder (from notebook 14) ─────
@dataclass
class PWLBracket:
    name: str
    breakpoints: List[float]
    slope_lo: List[float]; int_lo: List[float]
    slope_up: List[float]; int_up: List[float]
    @property
    def n_pieces(self): return len(self.breakpoints) - 1
    @property
    def domain(self):   return (self.breakpoints[0], self.breakpoints[-1])

def _line(p1, p2):
    s = (p2[1]-p1[1])/(p2[0]-p1[0]); return s, p1[1]-s*p1[0]
def _tan(f, df, x0):
    s = df(x0); return s, f(x0)-s*x0

def build_pwl_convex(f, df, a, b, n, name):
    bps = list(np.linspace(a, b, n+1))
    sl, il, su, iu = [], [], [], []
    for k in range(n):
        x_l, x_r = bps[k], bps[k+1]
        s_u, i_u = _line((x_l,f(x_l)),(x_r,f(x_r))); su.append(s_u); iu.append(i_u)
        s_l, i_l = _tan(f, df, 0.5*(x_l+x_r));       sl.append(s_l); il.append(i_l)
    return PWLBracket(name, bps, sl, il, su, iu)

def pwl_exp(a, b, n=8):       return build_pwl_convex(math.exp, math.exp, a, b, n, 'exp')
def pwl_inv_pos(a, b, n=8):
    assert a > 0
    return build_pwl_convex(lambda x: 1.0/x, lambda x: -1.0/(x*x), a, b, n, 'inv')

def add_pwl_bracket(model, x_var, y_var, bracket: PWLBracket,
                    big_M: float | None = None, prefix: str = ''):
    n = bracket.n_pieces; bps = bracket.breakpoints
    a, b = bps[0], bps[-1]
    if big_M is None:
        scope = max(
            max(abs(bracket.slope_lo[k])*(b-a)+abs(bracket.int_lo[k]) for k in range(n)),
            max(abs(bracket.slope_up[k])*(b-a)+abs(bracket.int_up[k]) for k in range(n)),
        )
        big_M = max(2.0*scope, 1.0)
    delta = [model.addVar(vtype=GRB.BINARY, name=f'{prefix}d_{k}') for k in range(n)]
    model.addConstr(gp.quicksum(delta) == 1, name=f'{prefix}dsum')
    for k in range(n):
        x_l, x_r = bps[k], bps[k+1]
        model.addConstr(x_var >= x_l - big_M*(1-delta[k]), name=f'{prefix}s{k}_xl')
        model.addConstr(x_var <= x_r + big_M*(1-delta[k]), name=f'{prefix}s{k}_xu')
        model.addConstr(y_var >= bracket.slope_lo[k]*x_var + bracket.int_lo[k]
                                 - big_M*(1-delta[k]), name=f'{prefix}s{k}_yl')
        model.addConstr(y_var <= bracket.slope_up[k]*x_var + bracket.int_up[k]
                                 + big_M*(1-delta[k]), name=f'{prefix}s{k}_yu')
    return delta

def add_mccormick(model, a_var, b_var, a_l, a_u, b_l, b_u, prefix='', y_lb=None, y_ub=None):
    """z = a·b McCormick envelope.  Returns Gurobi var z."""
    z_lo = min(a_l*b_l, a_l*b_u, a_u*b_l, a_u*b_u)
    z_up = max(a_l*b_l, a_l*b_u, a_u*b_l, a_u*b_u)
    if y_lb is not None: z_lo = max(z_lo, y_lb)
    if y_ub is not None: z_up = min(z_up, y_ub)
    z = model.addVar(lb=z_lo, ub=z_up, name=f'{prefix}z')
    model.addConstr(z >= a_l*b_var + b_l*a_var - a_l*b_l, name=f'{prefix}mc1')
    model.addConstr(z >= a_u*b_var + b_u*a_var - a_u*b_u, name=f'{prefix}mc2')
    model.addConstr(z <= a_u*b_var + b_l*a_var - a_u*b_l, name=f'{prefix}mc3')
    model.addConstr(z <= a_l*b_var + b_u*a_var - a_l*b_u, name=f'{prefix}mc4')
    return z

In [5]:
# ── IBP propagation through every attention sub-step ─────────────────────
def ibp_linear_np(x_l, x_u, W, b):
    """x_l, x_u: (..., E_in); W: (E_out, E_in); b: (E_out,) or None."""
    Wp = np.maximum(W, 0); Wn = np.minimum(W, 0)
    y_l = x_l @ Wp.T + x_u @ Wn.T
    y_u = x_u @ Wp.T + x_l @ Wn.T
    if b is not None:
        y_l = y_l + b; y_u = y_u + b
    return y_l, y_u

def ibp_attention_bounds(x_lo, x_up, attn):
    """
    Per-token sound IBP through attention.  Inputs are (N, E) numpy arrays.
    Returns dict of intermediate (lo, up) numpy arrays.
    """
    N, E = x_lo.shape
    H, D = attn.num_heads, attn.head_dim
    scale = attn.scale
    # extract weights
    Wq = attn.W_q.weight.detach().cpu().numpy()
    Wk = attn.W_k.weight.detach().cpu().numpy()
    Wv = attn.W_v.weight.detach().cpu().numpy()
    bv = attn.W_v.bias.detach().cpu().numpy() if attn.W_v.bias is not None else None
    Wo = attn.W_o.weight.detach().cpu().numpy()
    bo = attn.W_o.bias.detach().cpu().numpy() if attn.W_o.bias is not None else None

    # 1. Q, K, V (per-token linear, no Q/K bias per spec)
    Q_l, Q_u = ibp_linear_np(x_lo, x_up, Wq, None)             # (N, E)
    K_l, K_u = ibp_linear_np(x_lo, x_up, Wk, None)
    V_l, V_u = ibp_linear_np(x_lo, x_up, Wv, bv)
    # reshape to (H, N, D)
    def to_h(a): return a.reshape(N, H, D).transpose(1, 0, 2)
    Q_l, Q_u = to_h(Q_l), to_h(Q_u)
    K_l, K_u = to_h(K_l), to_h(K_u)
    V_l, V_u = to_h(V_l), to_h(V_u)

    # 2. scores S[h,i,j] = scale * Σ_d Q[h,i,d] · K[h,j,d]
    # Broadcast: Q (H, N, 1, D), K (H, 1, N, D)
    Q_e_l = Q_l[:, :, None, :]; Q_e_u = Q_u[:, :, None, :]
    K_e_l = K_l[:, None, :, :]; K_e_u = K_u[:, None, :, :]
    # element-wise interval product over (H, N, N, D)
    c1 = Q_e_l * K_e_l; c2 = Q_e_l * K_e_u; c3 = Q_e_u * K_e_l; c4 = Q_e_u * K_e_u
    P_l = np.minimum(np.minimum(c1, c2), np.minimum(c3, c4))
    P_u = np.maximum(np.maximum(c1, c2), np.maximum(c3, c4))
    S_l = scale * P_l.sum(axis=-1)
    S_u = scale * P_u.sum(axis=-1)                              # (H, N, N)

    # 3. row-wise stability shift = max over j of S_u[h,i,j]
    shift = S_u.max(axis=-1, keepdims=True)                     # (H, N, 1)
    S_sh_l = S_l - shift; S_sh_u = S_u - shift                  # both ≤ 0

    # 4. exp(S_shifted) — monotone increasing
    E_l = np.exp(S_sh_l); E_u = np.exp(S_sh_u)                  # (H, N, N), both in (0, 1]

    # 5. Sum_E[h,i] = Σ_j E[h,i,j]
    SumE_l = E_l.sum(axis=-1, keepdims=True)
    SumE_u = E_u.sum(axis=-1, keepdims=True)                    # (H, N, 1)
    # ensure strictly positive (otherwise softmax would be undefined)
    SumE_l = np.maximum(SumE_l, 1e-9)

    # 6. Inv = 1/Sum  — monotone decreasing
    Inv_l = 1.0 / SumE_u; Inv_u = 1.0 / SumE_l                  # (H, N, 1)

    # 7. A[h,i,j] = E[h,i,j] · Inv[h,i] — bilinear interval, clamp [0,1]
    Inv_b_l = np.broadcast_to(Inv_l, E_l.shape)
    Inv_b_u = np.broadcast_to(Inv_u, E_u.shape)
    cA1 = E_l*Inv_b_l; cA2 = E_l*Inv_b_u; cA3 = E_u*Inv_b_l; cA4 = E_u*Inv_b_u
    A_l = np.maximum(np.minimum(np.minimum(cA1,cA2),np.minimum(cA3,cA4)), 0.0)
    A_u = np.minimum(np.maximum(np.maximum(cA1,cA2),np.maximum(cA3,cA4)), 1.0)

    # 8. O[h,i,d] = Σ_j A[h,i,j] · V[h,j,d]
    A_e_l = A_l[..., None]; A_e_u = A_u[..., None]              # (H, N, N, 1)
    V_e_l = V_l[:, None, :, :]; V_e_u = V_u[:, None, :, :]      # (H, 1, N, D)
    cO1 = A_e_l*V_e_l; cO2 = A_e_l*V_e_u; cO3 = A_e_u*V_e_l; cO4 = A_e_u*V_e_u
    OP_l = np.minimum(np.minimum(cO1,cO2),np.minimum(cO3,cO4))
    OP_u = np.maximum(np.maximum(cO1,cO2),np.maximum(cO3,cO4))
    O_l = OP_l.sum(axis=2); O_u = OP_u.sum(axis=2)              # (H, N, D)

    # 9. concat heads: (N, E)
    O_cat_l = O_l.transpose(1, 0, 2).reshape(N, E)
    O_cat_u = O_u.transpose(1, 0, 2).reshape(N, E)

    # 10. output projection
    out_l, out_u = ibp_linear_np(O_cat_l, O_cat_u, Wo, bo)

    return dict(
        Q=(Q_l, Q_u), K=(K_l, K_u), V=(V_l, V_u),
        S=(S_l, S_u), shift=shift,
        S_shifted=(S_sh_l, S_sh_u), E=(E_l, E_u),
        SumE=(SumE_l, SumE_u), Inv=(Inv_l, Inv_u),
        A=(A_l, A_u), O=(O_l, O_u), out=(out_l, out_u),
    )

In [6]:
# ── MILP encoding of MHSA (single-batch, single layer) ───────────────────
def encode_mhsa(milp, x_vars, x_lo, x_up, attn,
                ibp_b: dict, n_pieces: int = 8, prefix: str = ''):
    """
    x_vars: nested list shape (N, E) of Gurobi vars (input tokens)
    x_lo, x_up: (N, E) numpy arrays of input bounds
    attn: PyTorch MHSA module (provides W_q, W_k, W_v, W_o, num_heads, head_dim, scale)
    ibp_b: precomputed bounds dict from ibp_attention_bounds()

    Returns out_vars: nested list shape (N, E) of Gurobi vars for attn(x).
    """
    N = len(x_vars); E = len(x_vars[0])
    H, D = attn.num_heads, attn.head_dim
    scale = attn.scale
    Wq = attn.W_q.weight.detach().cpu().numpy()
    Wk = attn.W_k.weight.detach().cpu().numpy()
    Wv = attn.W_v.weight.detach().cpu().numpy()
    bv = attn.W_v.bias.detach().cpu().numpy()  if attn.W_v.bias  is not None else np.zeros(E)
    Wo = attn.W_o.weight.detach().cpu().numpy()
    bo = attn.W_o.bias.detach().cpu().numpy()  if attn.W_o.bias  is not None else np.zeros(E)

    Q_l, Q_u = ibp_b['Q']; K_l, K_u = ibp_b['K']; V_l, V_u = ibp_b['V']
    S_l, S_u = ibp_b['S']
    shift = ibp_b['shift']
    Sh_l, Sh_u = ibp_b['S_shifted']
    E_l_b, E_u_b = ibp_b['E']
    SumE_l, SumE_u = ibp_b['SumE']
    Inv_l_b, Inv_u_b = ibp_b['Inv']
    A_l_b, A_u_b = ibp_b['A']

    # ── 1. Q, K, V projections (linear) ──
    # Per token i: q[i,e] = Σ_e' Wq[e,e']·x[i,e'] (no bias on Q/K)
    def linear_proj(W, b_arr, p):
        E_out = W.shape[0]
        out_h = []
        for i in range(N):
            row = []
            for e in range(E_out):
                expr = gp.quicksum(W[e, ep] * x_vars[i][ep] for ep in range(E)) + float(b_arr[e])
                v = milp.addVar(lb=-GRB.INFINITY, ub=GRB.INFINITY, name=f'{p}_{i}_{e}')
                milp.addConstr(v == expr, name=f'{p}_{i}_{e}_def')
                row.append(v)
            out_h.append(row)
        return out_h
    Q_vars = linear_proj(Wq, np.zeros(E), f'{prefix}Q')   # (N, E)
    K_vars = linear_proj(Wk, np.zeros(E), f'{prefix}K')
    V_vars = linear_proj(Wv, bv,           f'{prefix}V')

    # reshape (N, E) → (H, N, D) by view
    def to_heads(vars_NE):
        return [[[vars_NE[i][h*D + d] for d in range(D)] for i in range(N)] for h in range(H)]
    Qh = to_heads(Q_vars); Kh = to_heads(K_vars); Vh = to_heads(V_vars)

    # ── 2. scores S[h, i, j] = scale * Σ_d Q[h,i,d] · K[h,j,d] ──
    S_vars = [[[None]*N for _ in range(N)] for _ in range(H)]
    for h in range(H):
        for i in range(N):
            for j in range(N):
                # build sum of D bilinear products
                z_terms = []
                for d in range(D):
                    a_l = float(Q_l[h, i, d]); a_u = float(Q_u[h, i, d])
                    b_l = float(K_l[h, j, d]); b_u = float(K_u[h, j, d])
                    z = add_mccormick(milp, Qh[h][i][d], Kh[h][j][d],
                                      a_l, a_u, b_l, b_u,
                                      prefix=f'{prefix}qk_h{h}_i{i}_j{j}_d{d}_')
                    z_terms.append(z)
                s = milp.addVar(lb=float(S_l[h, i, j]) - 1e-3,
                                ub=float(S_u[h, i, j]) + 1e-3,
                                name=f'{prefix}S_h{h}_i{i}_j{j}')
                milp.addConstr(s == scale * gp.quicksum(z_terms),
                               name=f'{prefix}S_h{h}_i{i}_j{j}_def')
                S_vars[h][i][j] = s

    # ── 3. shift + exp ──
    A_vars = [[[None]*N for _ in range(N)] for _ in range(H)]
    for h in range(H):
        for i in range(N):
            sh_const = float(shift[h, i, 0])
            # 3a. exp variables for each j
            E_vars_row = []
            for j in range(N):
                sh_l = float(Sh_l[h, i, j]); sh_u = float(Sh_u[h, i, j])
                # PWL exp on [sh_l, sh_u]
                if sh_u - sh_l < 1e-9:
                    # degenerate range: encode as a constant
                    e_const = math.exp(0.5*(sh_l+sh_u))
                    e = milp.addVar(lb=e_const-1e-9, ub=e_const+1e-9,
                                    name=f'{prefix}E_h{h}_i{i}_j{j}')
                    milp.addConstr(e == e_const, name=f'{prefix}E_h{h}_i{i}_j{j}_pin')
                else:
                    br = pwl_exp(sh_l, sh_u, n=n_pieces)
                    # "S_shifted" = S - shift (constant); we don't materialise
                    # a variable for S_shifted — constraint exp via PWL on (S - shift)
                    s_sh = milp.addVar(lb=sh_l, ub=sh_u,
                                       name=f'{prefix}Ssh_h{h}_i{i}_j{j}')
                    milp.addConstr(s_sh == S_vars[h][i][j] - sh_const,
                                   name=f'{prefix}Ssh_h{h}_i{i}_j{j}_def')
                    e = milp.addVar(lb=float(E_l_b[h, i, j]) - 1e-6,
                                    ub=float(E_u_b[h, i, j]) + 1e-6,
                                    name=f'{prefix}E_h{h}_i{i}_j{j}')
                    add_pwl_bracket(milp, s_sh, e, br,
                                    prefix=f'{prefix}E_h{h}_i{i}_j{j}_')
                E_vars_row.append(e)

            # 3b. sum_e[h, i] = Σ_j e
            se_l = float(SumE_l[h, i, 0]); se_u = float(SumE_u[h, i, 0])
            sum_e = milp.addVar(lb=max(se_l, 1e-9), ub=se_u,
                                name=f'{prefix}SumE_h{h}_i{i}')
            milp.addConstr(sum_e == gp.quicksum(E_vars_row),
                           name=f'{prefix}SumE_h{h}_i{i}_def')

            # 3c. inv_e via PWL inv on [se_l, se_u]
            inv_l = float(Inv_l_b[h, i, 0]); inv_u = float(Inv_u_b[h, i, 0])
            if se_u - se_l < 1e-9:
                inv_const = 1.0 / (0.5*(se_l+se_u))
                inv_e = milp.addVar(lb=inv_const-1e-9, ub=inv_const+1e-9,
                                    name=f'{prefix}Inv_h{h}_i{i}')
                milp.addConstr(inv_e == inv_const,
                               name=f'{prefix}Inv_h{h}_i{i}_pin')
            else:
                br_inv = pwl_inv_pos(max(se_l, 1e-9), se_u, n=n_pieces)
                inv_e = milp.addVar(lb=inv_l - 1e-6, ub=inv_u + 1e-6,
                                    name=f'{prefix}Inv_h{h}_i{i}')
                add_pwl_bracket(milp, sum_e, inv_e, br_inv,
                                prefix=f'{prefix}Inv_h{h}_i{i}_')

            # 3d. A[h, i, j] = E · Inv via McCormick, clamp [0, 1]
            for j in range(N):
                a = add_mccormick(
                    milp, E_vars_row[j], inv_e,
                    float(E_l_b[h, i, j]), float(E_u_b[h, i, j]),
                    inv_l, inv_u,
                    prefix=f'{prefix}A_h{h}_i{i}_j{j}_',
                    y_lb=0.0, y_ub=1.0,
                )
                A_vars[h][i][j] = a

    # ── 4. O[h, i, d] = Σ_j A[h,i,j] · V[h,j,d] ──
    O_vars = [[[None]*D for _ in range(N)] for _ in range(H)]
    for h in range(H):
        for i in range(N):
            for d in range(D):
                z_terms = []
                for j in range(N):
                    a_l = float(A_l_b[h, i, j]); a_u = float(A_u_b[h, i, j])
                    v_l = float(V_l[h, j, d]);   v_u = float(V_u[h, j, d])
                    z = add_mccormick(milp, A_vars[h][i][j], Vh[h][j][d],
                                      a_l, a_u, v_l, v_u,
                                      prefix=f'{prefix}av_h{h}_i{i}_j{j}_d{d}_')
                    z_terms.append(z)
                o = milp.addVar(lb=-GRB.INFINITY, ub=GRB.INFINITY,
                                name=f'{prefix}O_h{h}_i{i}_d{d}')
                milp.addConstr(o == gp.quicksum(z_terms),
                               name=f'{prefix}O_h{h}_i{i}_d{d}_def')
                O_vars[h][i][d] = o

    # ── 5. concat heads + W_O ──
    # concat: O_cat[i, h*D + d] = O_vars[h][i][d]
    O_cat = [[O_vars[h][i][d] for h in range(H) for d in range(D)] for i in range(N)]
    out_vars = []
    for i in range(N):
        row = []
        for e in range(E):
            expr = gp.quicksum(Wo[e, ep] * O_cat[i][ep] for ep in range(E)) + float(bo[e])
            v = milp.addVar(lb=-GRB.INFINITY, ub=GRB.INFINITY,
                            name=f'{prefix}out_{i}_{e}')
            milp.addConstr(v == expr, name=f'{prefix}out_{i}_{e}_def')
            row.append(v)
        out_vars.append(row)
    return out_vars

In [7]:
# ── PyTorch reference MHSA (matches notebook 09 exactly) ─────────────────
class MHSA(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.embed_dim, self.num_heads = embed_dim, num_heads
        self.head_dim = embed_dim // num_heads
        self.scale    = self.head_dim ** -0.5
        self.W_q = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_k = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_v = nn.Linear(embed_dim, embed_dim, bias=True)
        self.W_o = nn.Linear(embed_dim, embed_dim, bias=True)
    def forward(self, x):
        B, N, C = x.shape; H, D = self.num_heads, self.head_dim
        q = self.W_q(x).view(B, N, H, D).transpose(1, 2)
        k = self.W_k(x).view(B, N, H, D).transpose(1, 2)
        v = self.W_v(x).view(B, N, H, D).transpose(1, 2)
        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        attn   = F.softmax(scores, dim=-1)
        out    = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, N, C)
        return self.W_o(out)

def true_attn(x_np, attn):
    x = torch.from_numpy(np.asarray(x_np, dtype=np.float32)).unsqueeze(0)  # (1, N, E)
    with torch.no_grad():
        return attn(x).squeeze(0).numpy()

In [8]:
# ── Soundness test driver ───────────────────────────────────────────────
def milp_attn_certified_range(x_lo, x_up, attn, n_pieces=8, time_limit=60.0):
    """
    Build MILP encoding for attention on input box [x_lo, x_up].
    Returns (out_min, out_max) arrays of shape (N, E) and timing.
    """
    if not HAS_GUROBI:
        return None, None, None
    N, E = x_lo.shape
    ibp_b = ibp_attention_bounds(x_lo, x_up, attn)

    m = gp.Model('attn_test')
    m.setParam('OutputFlag', 0); m.setParam('TimeLimit', time_limit)
    x_vars = [[m.addVar(lb=float(x_lo[i, e]), ub=float(x_up[i, e]), name=f'x_{i}_{e}')
               for e in range(E)] for i in range(N)]
    out_vars = encode_mhsa(m, x_vars, x_lo, x_up, attn, ibp_b, n_pieces=n_pieces)
    m.update()

    out_min = np.full((N, E), np.nan); out_max = np.full((N, E), np.nan)
    t0 = time.time()
    for i in range(N):
        for e in range(E):
            m.setObjective(out_vars[i][e], GRB.MINIMIZE); m.optimize()
            if m.Status not in (GRB.OPTIMAL, GRB.SUBOPTIMAL):
                raise RuntimeError(f'min[{i},{e}] status={m.Status}')
            out_min[i, e] = out_vars[i][e].X
            m.setObjective(out_vars[i][e], GRB.MAXIMIZE); m.optimize()
            if m.Status not in (GRB.OPTIMAL, GRB.SUBOPTIMAL):
                raise RuntimeError(f'max[{i},{e}] status={m.Status}')
            out_max[i, e] = out_vars[i][e].X
    return out_min, out_max, time.time() - t0


def soundness_test_attention(N: int, E: int, H: int, eps_in: float,
                              n_pieces: int = 8, K_pins: int = 8, seed: int = 0):
    torch.manual_seed(seed); np.random.seed(seed)
    attn = MHSA(E, H).eval()
    # Use small init so scores are in moderate range
    with torch.no_grad():
        for p in attn.parameters(): p.mul_(0.5)

    x_center = np.random.uniform(-0.5, 0.5, size=(N, E)).astype(np.float64)
    x_lo = x_center - eps_in
    x_up = x_center + eps_in

    out_min, out_max, solve_t = milp_attn_certified_range(
        x_lo, x_up, attn, n_pieces=n_pieces, time_limit=120)

    # Validate against PyTorch on K random pins in box + the center
    max_lo = max_up = 0.0
    pin_rng = np.random.default_rng(2024 + seed)
    test_xs = [x_center] + [x_lo + pin_rng.random(x_center.shape)*(x_up-x_lo)
                            for _ in range(K_pins)]
    for x_pin in test_xs:
        y_true = true_attn(x_pin, attn)
        max_lo = max(max_lo, float((out_min - y_true).max()))
        max_up = max(max_up, float((y_true - out_max).max()))

    return dict(
        N=N, E=E, H=H, eps_in=eps_in, n_pieces=n_pieces,
        solve_time_s = solve_t,
        mean_width   = float((out_max - out_min).mean()),
        max_width    = float((out_max - out_min).max()),
        max_lo_viol  = max_lo,
        max_up_viol  = max_up,
        sound        = (max_lo < 1e-4 and max_up < 1e-4),
    )

In [9]:
# ── Run soundness tests on tiny attention layers ────────────────────────
# The full ViT block (N=49, E=64, H=2) is exercised in notebook 17.
CONFIGS = [
    # (label, N, E, H, eps_in, n_pieces, seed)
    ('N=2 E=4  H=1', 2, 4, 1, 0.05, 8, 0),
    ('N=3 E=4  H=1', 3, 4, 1, 0.03, 8, 1),
    ('N=3 E=8  H=2', 3, 8, 2, 0.03, 8, 2),
    ('N=4 E=4  H=1', 4, 4, 1, 0.02, 8, 3),
]

if not HAS_GUROBI:
    print('Gurobi unavailable.')
    results = []
else:
    results = []
    print(f'{"label":15s} {"solve_s":>8s} {"max_w":>9s} {"mean_w":>9s} '
          f'{"lo_viol":>10s} {"up_viol":>10s}  status')
    print('─' * 80)
    for label, N, E, H, eps_in, npc, seed in CONFIGS:
        try:
            r = soundness_test_attention(N, E, H, eps_in,
                                         n_pieces=npc, K_pins=6, seed=seed)
        except Exception as e:
            print(f'{label}: FAILED ({e})')
            results.append(dict(label=label, error=str(e)))
            continue
        r['label'] = label
        results.append(r)
        status = 'SOUND' if r['sound'] else 'UNSOUND ✗'
        print(f'{label:15s} {r["solve_time_s"]:8.1f} {r["max_width"]:9.2e} '
              f'{r["mean_width"]:9.2e} {r["max_lo_viol"]:10.2e} '
              f'{r["max_up_viol"]:10.2e}  {status}')

    n_unsound = sum(1 for r in results if not r.get('sound', False))
    print(f'\n{len(results)-n_unsound}/{len(results)} configs sound')

label            solve_s     max_w    mean_w    lo_viol    up_viol  status
────────────────────────────────────────────────────────────────────────────────
Restricted license - for non-production use only - expires 2027-11-29
N=2 E=4  H=1         1.3  1.25e-02  1.14e-02   0.00e+00   0.00e+00  SOUND
N=3 E=4  H=1         2.9  7.96e-03  6.01e-03   0.00e+00   0.00e+00  SOUND
N=3 E=8  H=2        17.3  1.21e-02  1.02e-02   0.00e+00   0.00e+00  SOUND
N=4 E=4  H=1         8.1  1.03e-02  5.45e-03   0.00e+00   0.00e+00  SOUND

4/4 configs sound


In [10]:
# ── Save report ─────────────────────────────────────────────────────────
out_dir = Path('results/vit_p2'); out_dir.mkdir(parents=True, exist_ok=True)
(out_dir / 'milp_attention_report.json').write_text(
    json.dumps(results, indent=2, default=float))
print(f'Saved → {out_dir / "milp_attention_report.json"}')

try:
    drive_out = Path('/content/drive/My Drive/thesis-formal-verification/results/vit_p2')
    drive_out.mkdir(parents=True, exist_ok=True)
    (drive_out / 'milp_attention_report.json').write_text(
        (out_dir / 'milp_attention_report.json').read_text())
    print(f'Saved drive copy → {drive_out / "milp_attention_report.json"}')
except Exception as e:
    print(f'(skipping drive mirror: {e})')

print('\nPhase 4.3 complete: MHSA MILP encoding verified sound on tiny instances.')
print('Next: Phase 4.4–4.6 (full block + ViT verification; notebook 17).')

Saved → results/vit_p2/milp_attention_report.json
Saved drive copy → /content/drive/My Drive/thesis-formal-verification/results/vit_p2/milp_attention_report.json

Phase 4.3 complete: MHSA MILP encoding verified sound on tiny instances.
Next: Phase 4.4–4.6 (full block + ViT verification; notebook 17).


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

drive_out = Path('/content/drive/My Drive/thesis-formal-verification/results/vit_p2')
drive_out.mkdir(parents=True, exist_ok=True)
(drive_out / 'milp_attention_report.json').write_text(
    json.dumps(results, indent=2, default=float))
print(f'Saved drive copy → {drive_out / "milp_attention_report.json"}')